## Data Quality Assessment Overview

In this section, we evaluate the overall quality of the dataset before proceeding to analytical tasks. We begin by normalizing the nested JSON structure into a tabular format suitable for analysis, ensuring that all attributes are accessible at the column level.

We then assess key data quality dimensions, including completeness (missing values), structural consistency, duplication, and data type validity. These checks allow us to identify potential governance gaps, inconsistencies in data collection, and fields that may require cleaning or contextual interpretation before further analysis.

In [18]:
import json
import pandas as pd
from pathlib import Path

In [19]:
path = Path("..") / "Data" / "raw_credit_applications.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df.head()

,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
0,app_200,"{'full_name': 'Jerry Smith', 'email': 'jerry.s...","{'annual_income': 73000, 'credit_history_month...","[{'category': 'Shopping', 'amount': 480}, {'ca...","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,"{'full_name': 'Brandon Walker', 'email': 'bran...","{'annual_income': 78000, 'credit_history_month...","[{'category': 'Rent', 'amount': 608}, {'catego...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
2,app_215,"{'full_name': 'Scott Moore', 'email': 'scott.m...","{'annual_income': 61000, 'credit_history_month...","[{'category': 'Rent', 'amount': 109}]","{'loan_approved': True, 'interest_rate': 3.7, ...",NaN,vacation,NaN
3,app_024,"{'full_name': 'Thomas Lee', 'email': 'thomas.l...","{'annual_income': 103000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 575}]","{'loan_approved': True, 'interest_rate': 4.3, ...",NaN,NaN,NaN
4,app_184,"{'full_name': 'Brian Rodriguez', 'email': 'bri...","{'annual_income': 57000, 'credit_history_month...","[{'category': 'Entertainment', 'amount': 463}]","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN


The raw JSON dataset was successfully loaded and converted into a Pandas DataFrame. The structure confirms the presence of nested fields (e.g., applicant information, financial data, and decision details), which will require normalization before performing data quality analysis.

In [20]:
df.columns

Index(['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision',
       'processing_timestamp', 'loan_purpose', 'notes'],
      dtype='object')

In [21]:
flat = pd.json_normalize(data, sep=".")
flat.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN


In [22]:
flat["spending_behavior"].head()

0    [{'category': 'Shopping', 'amount': 480}, {'ca...
1    [{'category': 'Rent', 'amount': 608}, {'catego...
2                [{'category': 'Rent', 'amount': 109}]
3             [{'category': 'Fitness', 'amount': 575}]
4       [{'category': 'Entertainment', 'amount': 463}]
Name: spending_behavior, dtype: object

In [23]:
# Total rows
n_rows = flat.shape[0]

# Build missing summary table
missing_summary = pd.DataFrame({
    "values_present": flat.notna().sum(),
    "values_missing": flat.isna().sum(),
})

missing_summary["percent_missing"] = (
    missing_summary["values_missing"] / n_rows * 100
).round(2)

# Sort by most missing
missing_summary = missing_summary.sort_values(
    by="percent_missing", ascending=False
)

missing_summary

,values_present,values_missing,percent_missing
notes,2,500,99.60
financials.annual_salary,5,497,99.00
loan_purpose,50,452,90.04
processing_timestamp,62,440,87.65
decision.rejection_reason,210,292,58.17
decision.approved_amount,292,210,41.83
decision.interest_rate,292,210,41.83
financials.annual_income,497,5,1.00
applicant_info.ip_address,497,5,1.00
applicant_info.ssn,497,5,1.00


### Missing Values Analysis

The dataset contains 502 records and 21 variables. The missingness analysis reveals substantial gaps in several fields, particularly `notes` (99.6% missing), `financials.annual_salary` (99.0% missing), `loan_purpose` (90.0% missing), and `processing_timestamp` (87.6% missing). These fields appear either optional, inconsistently recorded, or poorly governed within the data collection process.

Decision-related variables such as `decision.rejection_reason` (58.2% missing) and `decision.approved_amount` / `decision.interest_rate` (41.8% missing) likely reflect structural missingness (e.g., rejection reason only exists for rejected loans). In contrast, core financial and applicant identification fields (e.g., income, SSN, email, credit history, debt-to-income ratio) show near-complete coverage, indicating stronger governance and validation controls for critical underwriting variables.